<a href="https://colab.research.google.com/github/mnuvunm/2026_tues_bigdatacomputing_class/blob/main/%EA%B8%B0%EB%A7%90%EA%B3%A0%EC%82%AC%20%EA%B3%BC%EC%A0%9C%20%EC%A0%9C%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q streamlit pyngrok scikit-learn pandas matplotlib seaborn

In [15]:
# =====================================================================
# 다항 회귀 및 릿지 규제 파이프라인 구축
# =====================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# ---------------------------------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------------------------------
url = "https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv"
df = pd.read_csv(url)

df.dropna(inplace=True)               # 결측치 제거
df.columns = df.columns.str.strip()   # 컬럼명 앞뒤 공백 제거 (KeyError 방지)

# ---------------------------------------------------------------------
# 2. 특성(Features) 및 타겟(Target) 분리
# ---------------------------------------------------------------------
# 조건: Schooling 제외, 최소 3개 이상 자율 선택
feature_names = ['BMI', 'GDP', 'Alcohol', 'Polio']
target_name = 'Life expectancy'

X = df[feature_names]
y = df[target_name]

# ---------------------------------------------------------------------
# 3. 데이터 분할 및 소규모 샘플링 (과대적합 유도용)
# ---------------------------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 훈련 데이터에서 50개만 무작위 추출
X_train = X_train_full.sample(n=50, random_state=42)
y_train = y_train_full.loc[X_train.index]

# ---------------------------------------------------------------------
# 4. 파이프라인(Pipeline) 기반 모델 3종 구축
# ---------------------------------------------------------------------
# 4-1. Model 1 (Linear): 기본 선형 회귀
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# 4-2. Model 2 (Poly): 3차 다항 회귀 (규제 없음 -> 과대적합 유도)
pipe_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('regressor', LinearRegression())
])

# 4-3. Model 3 (Ridge): 3차 다항 회귀 + 릿지 규제
pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('regressor', Ridge(alpha=1.0))
])

pipelines = {
    'Linear': pipe_linear,
    'Poly': pipe_poly,
    'Ridge': pipe_ridge
}

# ---------------------------------------------------------------------
# 5. 모델 학습, 평가 및 산출물 저장
# ---------------------------------------------------------------------
metrics = []

for name, pipe in pipelines.items():
    # 모델 학습
    pipe.fit(X_train, y_train)

    # 파이프라인 객체 저장 (.pkl)
    joblib.dump(pipe, f'{name}_model.pkl')

    # 예측 수행
    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)

    # 변환 후 특성 개수(Complexity) 확인
    if name == 'Linear':
        complexity = X_train.shape[1]
    else:
        complexity = pipe.named_steps['poly'].transform(X_train).shape[1]

    # 평가 지표 기록
    metrics.append({
        'Model': name,
        'Train R²': round(r2_score(y_train, pred_train), 4),
        'Test R²': round(r2_score(y_test, pred_test), 4),
        'Train MSE': round(mean_squared_error(y_train, pred_train), 2),
        'Test MSE': round(mean_squared_error(y_test, pred_test), 2),
        'Complexity': complexity
    })

# 지표 DataFrame 변환 및 저장
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv('metrics.csv', index=False)

print("✅ 성공: 데이터 전처리, 파이프라인 학습 및 평가 지표 저장이 완료되었습니다!")
display(metrics_df)

✅ 성공: 데이터 전처리, 파이프라인 학습 및 평가 지표 저장이 완료되었습니다!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PolynomialFeatures was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PolynomialFeatures was fitted without feature names
  warnings.warn(


,Model,Train R²,Test R²,Train MSE,Test MSE,Complexity
0,Linear,0.4845,0.2624,56.53,52.39,4
1,Poly,0.8658,-870.9327,14.71,61926.83,34
2,Ridge,0.7912,-0.1114,22.90,78.93,34


In [16]:
# Streamlit 앱 파일 생성 (app.py)
%%writefile app.py
# =====================================================================
# [웹 대시보드] WHO 기대수명 예측 및 과대적합/규제 비교
# =====================================================================

import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------
# 1. 페이지 기본 설정 및 리소스 로드
# ---------------------------------------------------------------------
st.set_page_config(page_title="Life Expectancy Prediction", layout="wide")

@st.cache_resource
def load_pipelines():
    """학습된 머신러닝 파이프라인 모델을 캐싱하여 로드합니다."""
    return {
        'Linear': joblib.load('Linear_model.pkl'),
        'Poly': joblib.load('Poly_model.pkl'),
        'Ridge': joblib.load('Ridge_model.pkl')
    }

@st.cache_data
def load_metrics():
    """저장된 모델 성능 지표 데이터를 캐싱하여 로드합니다."""
    return pd.read_csv('metrics.csv')

models = load_pipelines()
metrics_df = load_metrics()

# ---------------------------------------------------------------------
# 2. 대시보드 헤더 구성
# ---------------------------------------------------------------------
st.title("🌍 WHO 기대수명 예측 대시보드")
st.markdown("다항 회귀 및 릿지(Ridge) 규제 모델의 성능을 비교하고 실시간으로 기대수명을 예측해봅니다.")
st.markdown("---")

# ---------------------------------------------------------------------
# 3. 모델 성능 비교 화면 (과대적합 관찰)
# ---------------------------------------------------------------------
st.header("📊 1. 모델 성능 비교 (과대적합 관찰)")
col1, col2 = st.columns(2)

with col1:
    st.subheader("성능 평가지표 테이블")
    st.markdown("훈련 샘플 50개 환경에서의 지표입니다. 규제가 없는 Poly 모델의 붕괴를 확인하세요.")
    st.dataframe(metrics_df, use_container_width=True)

with col2:
    st.subheader("Test R² 점수 비교")
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=metrics_df, x='Model', y='Test R²', palette='magma', ax=ax)

    # 막대 그래프 위 수치 텍스트 표시
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=10, color='black', xytext=(0, 8),
                    textcoords='offset points')
    st.pyplot(fig)

st.markdown("---")

# ---------------------------------------------------------------------
# 4. 실시간 예측 UI 구성
# ---------------------------------------------------------------------
st.header("🔮 2. 실시간 기대수명 예측")

# 사이드바: 사용자 입력 컨트롤
st.sidebar.header("입력 변수 설정")
st.sidebar.markdown("슬라이더를 움직여 값을 조정해보세요.")

bmi = st.sidebar.slider("BMI (체질량 지수)", 1.0, 90.0, 40.0)
gdp = st.sidebar.slider("1인당 GDP (USD)", 1, 120000, 5000)
alcohol = st.sidebar.slider("알코올 소비량 (리터)", 0.0, 20.0, 5.0)
polio = st.sidebar.slider("Polio (소아마비 예방접종률 %)", 1.0, 99.0, 80.0)

selected_model_name = st.selectbox(
    "예측에 사용할 머신러닝 모델을 선택하세요:",
    ['Linear', 'Poly', 'Ridge']
)

# ---------------------------------------------------------------------
# 5. 실시간 동적 예측 수행 및 결과 출력
# ---------------------------------------------------------------------
# 사용자 입력을 DataFrame으로 변환
input_df = pd.DataFrame({
    'BMI': [bmi],
    'GDP': [gdp],
    'Alcohol': [alcohol],
    'Polio': [polio]
})

# 선택된 파이프라인으로 예측
selected_pipeline = models[selected_model_name]
prediction_result = selected_pipeline.predict(input_df)[0]

# 결과 화면 렌더링
st.success(f"현재 동작 중인 모델: **{selected_model_name}**")
st.markdown(f"""
<div style="background-color:#f0f2f6;padding:20px;border-radius:10px;text-align:center;">
    <h2 style="color:#1f77b4;margin:0;">예상 기대수명</h2>
    <h1 style="color:#d62728;font-size:50px;margin:0;">{prediction_result:.2f} 세</h1>
</div>
""", unsafe_allow_html=True)

Overwriting app.py


In [17]:
# [Cell 3] Streamlit 실행 및 ngrok 터널링
from pyngrok import ngrok
import subprocess
import time

ngrok.set_auth_token("2SbsEfdNUHDAUqWP9iavCT69lGP_86VKpea4dB2WKsMyCk1Qy")

# 기존 백그라운드에서 실행 중인 Streamlit 프로세스 종료 (충돌 방지)
!pkill streamlit

# Streamlit 앱을 백그라운드에서 실행 (포트 8501)
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# 서버가 켜질 때까지 3초 대기
time.sleep(3)

# ngrok을 통해 포트 8501 터널링
public_url = ngrok.connect(8501).public_url
print(f"✅ 웹 서비스가 성공적으로 배포되었습니다!")
print(f"👉 접속 링크: {public_url}")

✅ 웹 서비스가 성공적으로 배포되었습니다!
👉 접속 링크: https://7bf2-34-48-91-202.ngrok-free.app
